# Dataset Preprocessing

This notebook preprocesses the German Credit and HELOC datasets for the SHAP-enhanced fair credit scoring project. The workflow keeps the original raw files unchanged and saves model-ready train/test artifacts for later TabNet, SHAP, and fairness experiments.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

## 1. Load datasets

In [ ]:
german_path = DATA_DIR / "german_clean.csv"
heloc_path = DATA_DIR / "heloc_dataset_v1 (1).csv"

german_df = pd.read_csv(german_path)
heloc_df = pd.read_csv(heloc_path)

print(f"German Credit shape: {german_df.shape}")
print(f"HELOC shape: {heloc_df.shape}")
display(german_df.head())
display(heloc_df.head())

In [ ]:
summary = pd.DataFrame(
    {
        "dataset": ["German Credit", "HELOC"],
        "rows": [len(german_df), len(heloc_df)],
        "columns": [german_df.shape[1], heloc_df.shape[1]],
        "missing_values": [int(german_df.isna().sum().sum()), int(heloc_df.isna().sum().sum())],
    }
)
display(summary)

display(german_df["target"].value_counts().rename_axis("target").reset_index(name="count"))
display(heloc_df["RiskPerformance"].value_counts().rename_axis("RiskPerformance").reset_index(name="count"))

## 2. German Credit preprocessing

The German Credit dataset contains integer-coded categorical variables and a binary target. The target is kept as `1 = creditworthy` and `0 = not creditworthy`. Numeric columns are median-imputed and scaled. Categorical columns are most-frequent-imputed and one-hot encoded. Protected attributes are saved separately for fairness evaluation.

In [ ]:
german_target = "target"

german_numeric_features = [
    "loan_duration_months",
    "loan_amount",
    "installment_rate",
    "residence_duration",
    "age",
    "existing_credits",
    "number_of_dependents",
]

german_categorical_features = [
    col
    for col in german_df.columns
    if col not in german_numeric_features + [german_target]
]

german_features = german_numeric_features + german_categorical_features
X_german = german_df[german_features].copy()
y_german = german_df[german_target].astype(int).copy()

# Fairness attributes are kept outside the feature matrix for evaluation.
german_protected = pd.DataFrame(
    {
        "personal_status_sex": german_df["personal_status_sex"],
        "age_group": np.where(german_df["age"] < 26, "under_26", "26_and_over"),
    }
)

try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

german_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            german_numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("one_hot", one_hot_encoder),
                ]
            ),
            german_categorical_features,
        ),
    ],
    remainder="drop",
)

Xg_train, Xg_test, yg_train, yg_test, pg_train, pg_test = train_test_split(
    X_german,
    y_german,
    german_protected,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_german,
)

Xg_train_array = german_preprocessor.fit_transform(Xg_train)
Xg_test_array = german_preprocessor.transform(Xg_test)
german_feature_names = german_preprocessor.get_feature_names_out()

Xg_train_processed = pd.DataFrame(Xg_train_array, columns=german_feature_names, index=Xg_train.index)
Xg_test_processed = pd.DataFrame(Xg_test_array, columns=german_feature_names, index=Xg_test.index)

Xg_train_processed.to_csv(PROCESSED_DIR / "german_X_train.csv", index=False)
Xg_test_processed.to_csv(PROCESSED_DIR / "german_X_test.csv", index=False)
yg_train.to_frame("target").to_csv(PROCESSED_DIR / "german_y_train.csv", index=False)
yg_test.to_frame("target").to_csv(PROCESSED_DIR / "german_y_test.csv", index=False)
pg_train.reset_index(drop=True).to_csv(PROCESSED_DIR / "german_protected_train.csv", index=False)
pg_test.reset_index(drop=True).to_csv(PROCESSED_DIR / "german_protected_test.csv", index=False)
joblib.dump(german_preprocessor, MODEL_DIR / "german_preprocessor.joblib")

print("German preprocessing complete")
print(f"Train shape: {Xg_train_processed.shape}")
print(f"Test shape: {Xg_test_processed.shape}")
print(f"Saved files to: {PROCESSED_DIR}")

## 3. HELOC preprocessing

HELOC is numeric, but several columns use special negative values (`-9`, `-8`, `-7`) to represent missing or non-applicable bureau conditions. For modeling, these special codes are converted to missing values and median-imputed. Indicator columns are added first so the model can still learn whether a special code was originally present. The target is mapped as `Good = 1` and `Bad = 0`.

In [ ]:
heloc_special_codes = [-9, -8, -7]
heloc_feature_columns = [col for col in heloc_df.columns if col != "RiskPerformance"]

special_value_summary = (
    heloc_df[heloc_feature_columns]
    .isin(heloc_special_codes)
    .sum()
    .sort_values(ascending=False)
    .rename("special_code_count")
    .reset_index()
    .rename(columns={"index": "feature"})
)

display(special_value_summary.head(15))

In [ ]:
heloc = heloc_df.copy()
heloc["target"] = heloc["RiskPerformance"].map({"Bad": 0, "Good": 1}).astype(int)

for column in heloc_feature_columns:
    has_special_code = heloc[column].isin(heloc_special_codes)
    if has_special_code.any():
        heloc[f"{column}_special_code"] = has_special_code.astype(int)

heloc[heloc_feature_columns] = heloc[heloc_feature_columns].replace(heloc_special_codes, np.nan)

X_heloc = heloc.drop(columns=["RiskPerformance", "target"])
y_heloc = heloc["target"]

heloc_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_heloc,
    y_heloc,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_heloc,
)

Xh_train_array = heloc_preprocessor.fit_transform(Xh_train)
Xh_test_array = heloc_preprocessor.transform(Xh_test)

Xh_train_processed = pd.DataFrame(Xh_train_array, columns=X_heloc.columns, index=Xh_train.index)
Xh_test_processed = pd.DataFrame(Xh_test_array, columns=X_heloc.columns, index=Xh_test.index)

Xh_train_processed.to_csv(PROCESSED_DIR / "heloc_X_train.csv", index=False)
Xh_test_processed.to_csv(PROCESSED_DIR / "heloc_X_test.csv", index=False)
yh_train.to_frame("target").to_csv(PROCESSED_DIR / "heloc_y_train.csv", index=False)
yh_test.to_frame("target").to_csv(PROCESSED_DIR / "heloc_y_test.csv", index=False)
special_value_summary.to_csv(PROCESSED_DIR / "heloc_special_value_summary.csv", index=False)
joblib.dump(heloc_preprocessor, MODEL_DIR / "heloc_preprocessor.joblib")

print("HELOC preprocessing complete")
print(f"Train shape: {Xh_train_processed.shape}")
print(f"Test shape: {Xh_test_processed.shape}")
print(f"Saved files to: {PROCESSED_DIR}")

## 4. Validation checks

In [ ]:
checks = pd.DataFrame(
    [
        {
            "dataset": "German train",
            "rows": len(Xg_train_processed),
            "features": Xg_train_processed.shape[1],
            "missing_values": int(Xg_train_processed.isna().sum().sum()),
            "target_positive_rate": float(yg_train.mean()),
        },
        {
            "dataset": "German test",
            "rows": len(Xg_test_processed),
            "features": Xg_test_processed.shape[1],
            "missing_values": int(Xg_test_processed.isna().sum().sum()),
            "target_positive_rate": float(yg_test.mean()),
        },
        {
            "dataset": "HELOC train",
            "rows": len(Xh_train_processed),
            "features": Xh_train_processed.shape[1],
            "missing_values": int(Xh_train_processed.isna().sum().sum()),
            "target_positive_rate": float(yh_train.mean()),
        },
        {
            "dataset": "HELOC test",
            "rows": len(Xh_test_processed),
            "features": Xh_test_processed.shape[1],
            "missing_values": int(Xh_test_processed.isna().sum().sum()),
            "target_positive_rate": float(yh_test.mean()),
        },
    ]
)

display(checks)
assert checks["missing_values"].eq(0).all(), "Processed feature matrices should not contain missing values."
print("All validation checks passed.")